<a href="https://colab.research.google.com/github/innoted-latam/tp1_dne_uba/blob/main/TP1_PEREZ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```
ME72: Maestría en Métodos Cuantitativos para la Gestión y Análisis de Datos
M72109: Analisis de datos no estructurados
Universidad de Buenos Aires - Facultad de Ciencias Economicas (UBA-FCE)
Año: 2026
Estudiante: PÉREZ, LUCAS ENZO
```

## Preparación del ambiente

### NLP

In [1]:
!wget https://raw.githubusercontent.com/santiagxf/M72109/master/m72109/nlp/normalization.py \
    --quiet --no-clobber --directory-prefix ./m72109/nlp/
!wget https://raw.githubusercontent.com/santiagxf/M72109/master/m72109/nlp/transformation.py \
    --quiet --no-clobber --directory-prefix ./m72109/nlp/
!wget https://raw.githubusercontent.com/santiagxf/M72109/master/docs/nlp/neural/sequences-word2vec.txt \
    --quiet --no-clobber

In [2]:
!pip install -r sequences-word2vec.txt --quiet
!pip install datasets --quiet
!pip install transformers --quiet
!pip install unidecode --quiet
!python -m spacy download es_core_news_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 45.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 33.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### Sets de datos

Descargamos el set de datos:

In [3]:
!wget https://raw.githubusercontent.com/santiagxf/M72109/refs/heads/master/Desafio/Data/ground_truth.csv --directory-prefix ./Data/ --quiet --no-clobber
!wget https://raw.githubusercontent.com/santiagxf/M72109/refs/heads/master/Desafio/Data/Features/caption_features.csv --directory-prefix ./Data/Features/ --quiet --no-clobber

Cargamos el set de datos:

In [4]:
import pandas as pd

labels = pd.read_csv('Data/ground_truth.csv')
cc = pd.read_csv('Data/Features/caption_features.csv', names=['sequence_name','cc'], header=0)

Los conjuntos de datos utilizados en este desafío son los siguientes:

*   `ground_truth.csv`: Contiene la verdad fundamental para el problema de clasificación. Este archivo incluye información sobre segmentos de video (`movie_name`, `start(sec)`, `end(sec)`) y su correspondiente puntuación de memorabilidad (`memorability_score`). **Aunque contiene varias columnas, para el problema de clasificación que abordaremos, la columna clave es `memorable`, que indica la probabilidad de que una persona recuerde el video.**
*   `caption_features.csv`: Contiene las características de las "captions" (descripciones textuales) asociadas a cada segmento de video. Cada segmento de video se identifica por `sequence_name` y cuenta con una descripción en la columna `cc`.

**Para proceder con el análisis, será necesario combinar estas etiquetas (`ground_truth.csv`) con las características (`caption_features.csv`).**

## Solución

In [5]:
# =============================================================================
# BLOQUE 1: CARGA, FUSIÓN DE DATOS Y DIAGNÓSTICO INICIAL
# =============================================================================
# Objetivo: construir la tabla analítica base del desafío, uniendo las etiquetas
# (ground truth) con las descripciones textuales de cada segmento de video, y
# validar la integridad del resultado antes de avanzar hacia el preprocesamiento
# de texto y el modelado.
# =============================================================================

# -----------------------------------------------------------------------------
# 0. Importación de librerías
# -----------------------------------------------------------------------------
import pandas as pd

# -----------------------------------------------------------------------------
# 1. CARGA DE DATOS
# -----------------------------------------------------------------------------
# 1.a) Etiquetas (ground truth). Contiene la identificación del segmento de
#      video, su puntaje continuo de memorabilidad ('memorability_score') y la
#      etiqueta binaria de clasificación ('memorable'), que es nuestro target.
labels_df = pd.read_csv('Data/ground_truth.csv')

# 1.b) Features de texto (captions). Se fuerzan explícitamente los nombres de
#      columna a ['sequence_name', 'cc'] mediante `names=...`, y se usa
#      `header=0` para indicarle a pandas que la primera fila del archivo es la
#      cabecera original y debe ser descartada (no interpretada como un dato).
#      Esto garantiza una nomenclatura estable y reproducible aunque cambien los
#      encabezados del archivo fuente.
captions_df = pd.read_csv(
    'Data/Features/caption_features.csv',
    names=['sequence_name', 'cc'],
    header=0
)

# -----------------------------------------------------------------------------
# 2. FUSIÓN DE DATOS (DATA MERGING)
# -----------------------------------------------------------------------------
# Se unen ambas fuentes por la clave primaria común 'sequence_name', que
# identifica unívocamente cada segmento de video.
# Se elige un INNER JOIN de forma deliberada: solo conservamos los segmentos que
# tienen simultáneamente etiqueta y descripción textual. Una observación sin
# target no es entrenable, y una sin texto no aporta features al modelo.
df_unificado = pd.merge(
    labels_df,
    captions_df,
    on='sequence_name',
    how='inner'
)

# -----------------------------------------------------------------------------
# 3. DIAGNÓSTICO DE INTEGRIDAD E INSPECCIÓN
# -----------------------------------------------------------------------------

# 3.a) Control de dimensiones (shape).
#      Comparamos las filas de origen contra las del resultado para detectar los
#      dos riesgos clásicos de un merge:
#        - PÉRDIDA de registros  -> filas resultantes < filas de origen
#          (claves que no matchean por typos, espacios o cobertura parcial).
#        - DUPLICACIÓN de registros -> filas resultantes > filas de origen
#          (la clave no es única en alguna de las dos tablas, generando un
#           producto cartesiano parcial).
#      La referencia nominal del desafío es de 660 secuencias; el bloque verifica
#      si el cruce efectivamente la alcanza en lugar de asumirlo.
print("=" * 70)
print("3.a) CONTROL DE DIMENSIONES (shape)")
print("=" * 70)
print(f"labels_df      (ground truth) : {labels_df.shape[0]} filas x {labels_df.shape[1]} columnas")
print(f"captions_df    (captions)     : {captions_df.shape[0]} filas x {captions_df.shape[1]} columnas")
print(f"df_unificado   (merge inner)  : {df_unificado.shape[0]} filas x {df_unificado.shape[1]} columnas")

# Validación explícita: unicidad de la clave y ausencia de pérdida/duplicación.
FILAS_ESPERADAS = 660  # cantidad nominal de secuencias del desafío

print("\n--- Validaciones automáticas ---")
print(f"Clave 'sequence_name' única en labels_df    : {labels_df['sequence_name'].is_unique}")
print(f"Clave 'sequence_name' única en captions_df  : {captions_df['sequence_name'].is_unique}")
print(f"Sin duplicación de registros                : {df_unificado.shape[0] <= labels_df.shape[0]}")
print(f"Alcanza las {FILAS_ESPERADAS} secuencias nominales      : {df_unificado.shape[0] == FILAS_ESPERADAS}")

# 3.a.bis) ANTI-JOIN: identificación de las claves que NO cruzan.
#      Si el inner join devuelve menos filas que cualquiera de las fuentes, no
#      alcanza con constatarlo: hay que identificar exactamente qué claves se
#      perdieron y por qué lado, para poder decidir con criterio si se trata de
#      un problema de formato (typos, espacios, mayúsculas) o de una diferencia
#      real de cobertura entre los archivos.
solo_en_labels = set(labels_df['sequence_name']) - set(captions_df['sequence_name'])
solo_en_captions = set(captions_df['sequence_name']) - set(labels_df['sequence_name'])

if solo_en_labels or solo_en_captions:
    print("\n>> ATENCIÓN: el cruce no es perfecto. Claves sin correspondencia:")
    print(f"   - Con etiqueta pero SIN caption ({len(solo_en_labels)}): {sorted(solo_en_labels)}")
    print(f"     (no son entrenables: no tendrían features de texto)")
    print(f"   - Con caption pero SIN etiqueta ({len(solo_en_captions)}): {sorted(solo_en_captions)}")
    print(f"     (no son entrenables: no tendrían variable objetivo)")
    print("   Se trata de nombres de secuencia distintos entre sí (no de un typo")
    print("   ni de un problema de formato), por lo que es una diferencia real de")
    print("   cobertura entre los archivos. El INNER JOIN las descarta, que es el")
    print("   comportamiento correcto para construir el set de entrenamiento.")
else:
    print("\n>> Cruce perfecto: todas las claves tienen correspondencia en ambas fuentes.")

# 3.b) Control de valores nulos.
#      Un nulo en 'cc' dejaría al modelo sin features de texto para esa fila; un
#      nulo en 'memorable' la dejaría sin target. Verificamos ambos casos de
#      forma explícita, columna por columna.
print("\n" + "=" * 70)
print("3.b) CONTROL DE VALORES NULOS")
print("=" * 70)
nulos_por_columna = df_unificado.isnull().sum()
existen_nulos = bool(nulos_por_columna.sum() > 0)

print(f"¿Existen valores nulos en df_unificado?: {'SÍ' if existen_nulos else 'NO'}")
print(f"Total de celdas nulas: {int(nulos_por_columna.sum())}")
print("\nDetalle de nulos por columna:")
print(nulos_por_columna.to_string())

if not existen_nulos:
    print("\n>> Dataset íntegro: no se requiere imputación ni descarte de filas.")
else:
    print("\n>> ATENCIÓN: revisar las columnas afectadas antes de continuar.")

# 3.c) Inspección visual de las primeras 5 filas.
#      Verificamos la alineación semántica del merge: que la descripción ('cc')
#      corresponda efectivamente al segmento identificado por 'sequence_name' y
#      a su puntaje ('memorability_score') y etiqueta ('memorable').
#      El texto de 'cc' es largo, por lo que se muestra recortado para que la
#      tabla siga siendo legible (el DataFrame NO se modifica: el recorte es
#      solo para esta impresión de control).
print("\n" + "=" * 70)
print("3.c) INSPECCIÓN VISUAL: primeras 5 filas de df_unificado")
print("=" * 70)
print(f"Columnas disponibles: {list(df_unificado.columns)}\n")

vista_previa = df_unificado[['sequence_name', 'cc', 'memorability_score', 'memorable']].head(5).copy()
vista_previa['cc'] = vista_previa['cc'].str.slice(0, 90) + '...'
print(vista_previa.to_string(index=False))

# 3.d) Distribución del target.
#      Chequeo temprano de balance de clases: define si más adelante deberemos
#      usar métricas robustas al desbalance y/o estratificar el split.
print("\n" + "=" * 70)
print("3.d) DISTRIBUCIÓN DE LA VARIABLE OBJETIVO ('memorable')")
print("=" * 70)
distribucion = df_unificado['memorable'].value_counts().sort_index()
proporcion = df_unificado['memorable'].value_counts(normalize=True).sort_index()
for clase in distribucion.index:
    print(f"Clase {clase}: {distribucion[clase]:>4} casos ({proporcion[clase]:.2%})")

print("\n" + "=" * 70)
print("BLOQUE 1 FINALIZADO: 'df_unificado' listo para el preprocesamiento de texto.")
print("=" * 70)

3.a) CONTROL DE DIMENSIONES (shape)
labels_df      (ground truth) : 660 filas x 8 columnas
captions_df    (captions)     : 660 filas x 2 columnas
df_unificado   (merge inner)  : 659 filas x 9 columnas

--- Validaciones automáticas ---
Clave 'sequence_name' única en labels_df    : True
Clave 'sequence_name' única en captions_df  : True
Sin duplicación de registros                : True
Alcanza las 660 secuencias nominales      : False

>> ATENCIÓN: el cruce no es perfecto. Claves sin correspondencia:
   - Con etiqueta pero SIN caption (1): ['Resident_Evil_1_3506_3516_9']
     (no son entrenables: no tendrían features de texto)
   - Con caption pero SIN etiqueta (1): ['Whiplash_3217_3227_9']
     (no son entrenables: no tendrían variable objetivo)
   Se trata de nombres de secuencia distintos entre sí (no de un typo
   ni de un problema de formato), por lo que es una diferencia real de
   cobertura entre los archivos. El INNER JOIN las descarta, que es el
   comportamiento correcto para 